In [1]:
# Lab 4 - APIs & Static Web Scraping

#Topic: Books

#API: Open Library Search API
#Webpage: Books to Scrape

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
api_url = "https://openlibrary.org/search.json"

In [4]:
web_url = "https://books.toscrape.com/"

In [5]:
params = {
    "q": "python programming",
    "fields": "key,title,author_name,first_publish_year,edition_count,language",
    "limit": 30
}

In [6]:
response = requests.get(
    api_url,
    params=params,
    timeout=20
)

In [7]:
response.status_code

200

In [8]:
api_json = response.json()

In [9]:
type(api_json)

dict

In [10]:
api_json.keys()

dict_keys(['numFound', 'start', 'numFoundExact', 'num_found', 'documentation_url', 'q', 'offset', 'docs'])

In [11]:
api_json["docs"]

[{'author_name': ['R. Nageswara Rao'],
  'edition_count': 5,
  'first_publish_year': 2016,
  'key': '/works/OL30906747W',
  'language': ['eng', 'hin'],
  'title': 'Core Python Programming'},
 {'author_name': ['Justin Seitz', 'Tim Arnold'],
  'edition_count': 3,
  'first_publish_year': 2014,
  'key': '/works/OL19547345W',
  'language': ['eng', 'pol'],
  'title': 'Black Hat Python'},
 {'author_name': ['Wesley Chun'],
  'edition_count': 2,
  'first_publish_year': 2006,
  'key': '/works/OL12409294W',
  'language': ['eng'],
  'title': 'Core Python programming'},
 {'author_name': ['Reema Thareja'],
  'edition_count': 2,
  'first_publish_year': 2019,
  'key': '/works/OL26755050W',
  'language': ['eng'],
  'title': 'Python Programming'},
 {'author_name': ['John M. Zelle'],
  'edition_count': 2,
  'first_publish_year': 2003,
  'key': '/works/OL6037340W',
  'language': ['eng'],
  'title': 'Python programming'},
 {'author_name': ['Joshua Welsh'],
  'edition_count': 1,
  'first_publish_year': 2017

In [12]:
len(api_json["docs"])

30

In [13]:
api_json["docs"][0]

{'author_name': ['R. Nageswara Rao'],
 'edition_count': 5,
 'first_publish_year': 2016,
 'key': '/works/OL30906747W',
 'language': ['eng', 'hin'],
 'title': 'Core Python Programming'}

In [14]:
api_books = []

In [15]:
for book in api_json["docs"]:
    api_books.append({
        "Work_ID": book.get("key"),
        "Title": book.get("title"),
        "Author": ", ".join(book.get("author_name", []))
                  if book.get("author_name") else None,
        "First_Publish_Year": book.get("first_publish_year"),
        "Edition_Count": book.get("edition_count"),
        "Language": ", ".join(book.get("language", [])[:3])
                    if book.get("language") else None
    })

In [16]:
api_df = pd.DataFrame(api_books)

In [17]:
api_df.head()

,Work_ID,Title,Author,First_Publish_Year,Edition_Count,Language
0,/works/OL30906747W,Core Python Programming,R. Nageswara Rao,2016,5,"eng, hin"
1,/works/OL19547345W,Black Hat Python,"Justin Seitz, Tim Arnold",2014,3,"eng, pol"
2,/works/OL12409294W,Core Python programming,Wesley Chun,2006,2,eng
3,/works/OL26755050W,Python Programming,Reema Thareja,2019,2,eng
4,/works/OL6037340W,Python programming,John M. Zelle,2003,2,eng


In [18]:
api_df.shape

(30, 6)

In [19]:
api_df.columns

Index(['Work_ID', 'Title', 'Author', 'First_Publish_Year', 'Edition_Count',
       'Language'],
      dtype='object')

In [20]:
api_df.dtypes

,0
Work_ID,object
Title,object
Author,object
First_Publish_Year,int64
Edition_Count,int64
Language,object


In [21]:
api_df.isnull().sum()

,0
Work_ID,0
Title,0
Author,0
First_Publish_Year,0
Edition_Count,0
Language,3


In [22]:
api_df.duplicated().sum()

np.int64(0)

In [23]:
api_df["Work_ID"].duplicated().sum()

np.int64(0)

In [24]:
api_df = api_df.drop_duplicates(
    subset="Work_ID"
)

In [25]:
api_df = api_df.reset_index(drop=True)

In [26]:
api_df["Author"] = api_df["Author"].fillna("Unknown")

In [27]:
api_df["Language"] = api_df["Language"].fillna("Unknown")

In [28]:
api_df["First_Publish_Year"] = pd.to_numeric(
    api_df["First_Publish_Year"],
    errors="coerce"
).astype("Int64")

In [29]:
api_df["Edition_Count"] = pd.to_numeric(
    api_df["Edition_Count"],
    errors="coerce"
).astype("Int64")

In [30]:
api_df.dtypes

,0
Work_ID,object
Title,object
Author,object
First_Publish_Year,Int64
Edition_Count,Int64
Language,object


In [31]:
api_df.isnull().sum()

,0
Work_ID,0
Title,0
Author,0
First_Publish_Year,0
Edition_Count,0
Language,0


In [32]:
api_df.duplicated().sum()

np.int64(0)

In [33]:
api_df.head()

,Work_ID,Title,Author,First_Publish_Year,Edition_Count,Language
0,/works/OL30906747W,Core Python Programming,R. Nageswara Rao,2016,5,"eng, hin"
1,/works/OL19547345W,Black Hat Python,"Justin Seitz, Tim Arnold",2014,3,"eng, pol"
2,/works/OL12409294W,Core Python programming,Wesley Chun,2006,2,eng
3,/works/OL26755050W,Python Programming,Reema Thareja,2019,2,eng
4,/works/OL6037340W,Python programming,John M. Zelle,2003,2,eng


In [34]:
api_df.to_csv(
    "openlibrary_books.csv",
    index=False
)

In [35]:
pd.read_csv("openlibrary_books.csv").head()

,Work_ID,Title,Author,First_Publish_Year,Edition_Count,Language
0,/works/OL30906747W,Core Python Programming,R. Nageswara Rao,2016,5,"eng, hin"
1,/works/OL19547345W,Black Hat Python,"Justin Seitz, Tim Arnold",2014,3,"eng, pol"
2,/works/OL12409294W,Core Python programming,Wesley Chun,2006,2,eng
3,/works/OL26755050W,Python Programming,Reema Thareja,2019,2,eng
4,/works/OL6037340W,Python programming,John M. Zelle,2003,2,eng


In [36]:
web_response = requests.get(
    web_url,
    timeout=20
)

In [37]:
web_response.status_code

200

In [38]:
soup = BeautifulSoup(
    web_response.text,
    "html.parser"
)

In [39]:
book_cards = soup.select(
    "article.product_pod"
)

In [40]:
len(book_cards)

20

In [41]:
book_cards[5]

<article class="product_pod">
<div class="image_container">
<a href="catalogue/the-requiem-red_995/index.html"><img alt="The Requiem Red" class="thumbnail" src="media/cache/68/33/68339b4c9bc034267e1da611ab3b34f8.jpg"/></a>
</div>
<p class="star-rating One">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/the-requiem-red_995/index.html" title="The Requiem Red">The Requiem Red</a></h3>
<div class="product_price">
<p class="price_color">Â£22.65</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>

In [42]:
scraped_books = []

In [43]:
for book in book_cards:
    title = book.h3.a["title"]

    price = book.select_one(
        ".price_color"
    ).get_text(strip=True)

    availability = book.select_one(
        ".availability"
    ).get_text(strip=True)

    rating = book.select_one(
        ".star-rating"
    )["class"][1]

    scraped_books.append({
        "Title": title,
        "Price": price,
        "Availability": availability,
        "Rating": rating
    })

In [44]:
web_df = pd.DataFrame(scraped_books)

In [45]:
web_df.head()

,Title,Price,Availability,Rating
0,A Light in the Attic,Â£51.77,In stock,Three
1,Tipping the Velvet,Â£53.74,In stock,One
2,Soumission,Â£50.10,In stock,One
3,Sharp Objects,Â£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five


In [46]:
web_df.shape

(20, 4)

In [47]:
web_df["Price_GBP"] = (
    web_df["Price"]
    .str.replace(r"[^\d.]", "", regex=True)
)

In [48]:
web_df["Price_GBP"] = pd.to_numeric(
    web_df["Price_GBP"],
    errors="coerce"
)

In [49]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [50]:
web_df["Rating_Number"] = (
    web_df["Rating"]
    .map(rating_map)
)

In [51]:
web_df.head()

,Title,Price,Availability,Rating,Price_GBP,Rating_Number
0,A Light in the Attic,Â£51.77,In stock,Three,51.77,3
1,Tipping the Velvet,Â£53.74,In stock,One,53.74,1
2,Soumission,Â£50.10,In stock,One,50.10,1
3,Sharp Objects,Â£47.82,In stock,Four,47.82,4
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,54.23,5


In [52]:
web_df.isnull().sum()

,0
Title,0
Price,0
Availability,0
Rating,0
Price_GBP,0
Rating_Number,0


In [53]:
web_df.duplicated().sum()

np.int64(0)

In [54]:
web_df["Title"].duplicated().sum()

np.int64(0)

In [55]:
web_df.dtypes

,0
Title,object
Price,object
Availability,object
Rating,object
Price_GBP,float64
Rating_Number,int64


In [56]:
api_df.shape

(30, 6)

In [57]:
api_df.columns

Index(['Work_ID', 'Title', 'Author', 'First_Publish_Year', 'Edition_Count',
       'Language'],
      dtype='object')

In [58]:
api_df.dtypes

,0
Work_ID,object
Title,object
Author,object
First_Publish_Year,Int64
Edition_Count,Int64
Language,object


In [59]:
api_df.isnull().sum()

,0
Work_ID,0
Title,0
Author,0
First_Publish_Year,0
Edition_Count,0
Language,0


In [60]:
api_df.duplicated().sum()

np.int64(0)

In [61]:
web_df.shape

(20, 6)

In [62]:
web_df.columns

Index(['Title', 'Price', 'Availability', 'Rating', 'Price_GBP',
       'Rating_Number'],
      dtype='object')

In [63]:
web_df.dtypes

,0
Title,object
Price,object
Availability,object
Rating,object
Price_GBP,float64
Rating_Number,int64


In [64]:
web_df.isnull().sum()

,0
Title,0
Price,0
Availability,0
Rating,0
Price_GBP,0
Rating_Number,0


In [65]:
web_df.duplicated().sum()

np.int64(0)

In [66]:
api_df["First_Publish_Year"].describe()

,First_Publish_Year
count,30.0
mean,2013.766667
std,6.662729
min,1996.0
25%,2011.5
50%,2016.0
75%,2018.0
max,2022.0


In [67]:
api_df["Edition_Count"].describe()

,Edition_Count
count,30.0
mean,3.433333
std,4.38401
min,1.0
25%,1.0
50%,2.0
75%,4.75
max,23.0


In [68]:
api_df.loc[
    api_df["Edition_Count"].idxmax(),
    ["Title", "Author", "Edition_Count"]
]

,6
Title,Learning Python
Author,"Mark Lutz, David Ascher"
Edition_Count,23


In [69]:
web_df["Price_GBP"].mean()

np.float64(38.048500000000004)

In [70]:
web_df["Price_GBP"].min()

13.99

In [71]:
web_df["Price_GBP"].max()

57.25

In [72]:
web_df["Rating_Number"].value_counts().sort_index()

,count
Rating_Number,
1,6
2,3
3,3
4,4
5,4


In [73]:
web_df.loc[
    web_df["Price_GBP"].idxmax(),
    ["Title", "Price_GBP", "Rating_Number"]
]

,15
Title,Our Band Could Be Your Life: Scenes from the A...
Price_GBP,57.25
Rating_Number,3


In [74]:
web_df.loc[
    web_df["Price_GBP"].idxmin(),
    ["Title", "Price_GBP", "Rating_Number"]
]

,10
Title,"Starving Hearts (Triangular Trade Trilogy, #1)"
Price_GBP,13.99
Rating_Number,2
